In [ ]:
import numpy as np
import pycountry
import json
from numpyro import distributions as dist
from numpyro import infer
from jax import random
from datetime import timedelta

from emu_renewal.renew import MultiStrainModel
from emu_renewal.process import CosineMultiCurve
from emu_renewal.distributions import GammaDens
from emu_renewal.calibration import StandardCalib
from emu_renewal.run import log, find_run_start_time, find_run_end_time, gather_targets, collate_targets, find_variant_seeds
from emu_renewal.inputs import DATE_FORMAT, BASE_PATH, DATA_PATH, get_indicator_series_from_who_data, \
    get_country_vacc_data, get_standard_targets, get_country_vars, \
    get_worldbank_national_pop, get_country_mobility, get_standard_priors
from emu_renewal.run import store_outputs

In [ ]:
analysis_name = "experiment_1"
country = "AUT"
mob_analysis_type = "no_mob"
init_duration = 50
seed_duration = 10
proc_update_freq = 7
num_chains = 4
iterations = 100
prog_bar = True
strain_seeding_rate = 100.0
seed_duration = 10

In [ ]:
initial_countries = json.load(open(DATA_PATH / f"config/countries.json", "r"))
all_countries = initial_countries["admissions"] + initial_countries["occupancy"]
hosp_out, hosp_out_name = ("Daily hospital occupancy", "occupancy") if country in initial_countries["occupancy"] else ("Weekly new hospital admissions", "admissions")

log(f"\n________________________\nRunning job at {analysis_name}")
iso3 = pycountry.countries.lookup(country).alpha_3
log(f"Country: {iso3}")
log(f"Mobility approach: {mob_analysis_type}")
pop = get_worldbank_national_pop(iso3)
data_start = find_run_start_time(iso3, pop, 2e-6)
most_extreme_prop = 0.05
end_time = find_run_end_time(country, most_extreme_prop)
cases_target, hosp_target, deaths_target, seroprev_target, prealpha_prop = gather_targets(iso3, data_start, end_time, 10, hosp_out)
targets = collate_targets(cases_target, deaths_target, hosp_target, hosp_out_name, seroprev_target, most_extreme_prop, prealpha_prop, data_start, end_time)
run_start = data_start - timedelta(40)
log(f"Running from {run_start.strftime(DATE_FORMAT)} with data starting from {data_start.strftime(DATE_FORMAT)}")
log(f"Running to {end_time.strftime(DATE_FORMAT)}")
seed_times = find_variant_seeds(0.5, prealpha_prop, run_start)
seed_times = [run_start] + seed_times
mobility = get_country_mobility(iso3)
priors = get_standard_priors()
model = MultiStrainModel(
    pop,
    run_start,
    end_time,
    proc_update_freq,
    CosineMultiCurve(),
    GammaDens(),
    init_duration,
    np.zeros(init_duration),
    GammaDens(),
    GammaDens(),
    ["eu", "alpha"],
    "eu",
    seed_times,
    strain_seeding_rate,
    mobility[mob_analysis_type].dropna(),
    seed_duration,
)

In [ ]:
calib = StandardCalib(model, priors, targets, proc_dispersion=dist.HalfNormal(0.5))
kernel = infer.NUTS(calib.calibration, dense_mass=True, init_strategy=calib.custom_init(radius=0.1))
mcmc = infer.MCMC(kernel, num_chains=num_chains, num_samples=iterations, num_warmup=iterations, progress_bar=prog_bar)
mcmc.run(random.PRNGKey(0), extra_fields=["potential_energy"])
storage_path = BASE_PATH / "outputs" / analysis_name / country / mob_analysis_type
storage_path.mkdir(parents=True, exist_ok=True)
log(f"Writing to: {storage_path}")
store_outputs(storage_path, model, calib, mcmc)